# Figure 2d — *GABBR1* sashimi plot inputs

Sashimi plot of a differentially spliced unproductive event in *GABBR1* across
GTEx samples. The unproductive event corresponds to an exon skipping event.

This notebook does not draw the panel itself: it prepares the inputs the
genome-tracks tool consumes. For each gene it writes a links BED of per-tissue
intron PSI (`CreateBedForSashimiLinks`) and then a tracks configuration
(`prepareGenomeTracks`), across the same ten tissues shown in Fig. 2e.

The *GABBR1* panel is the published Fig. 2d; the other genes prepared here
(NOC2L, ABCA5, MRTO4, CNNM3, SFT2D1, AKAP8L, DLG4) feed supplementary sashimi
panels.

Outputs go to `code/results/ds_sashimi_plots/`: `ds_links/<GENE>.bed`,
`ds_tracks/`, and the rendered plots in `ds_plots/`.

In [18]:
import pandas as pd
import numpy as np
import vcf
from matplotlib import pyplot as plt
import gzip

In [2]:
import tabix

# Define a function to read the content
def run_tabix(tabix_file, chrom, start, end, file_type = 'other'):
    tabix_obj = tabix.open(tabix_file)
    # Fetch data from the tabix file in the desired range
    result = tabix_obj.query(chrom, start, end)
    
    rows = [line for line in result]
    
    with gzip.open(tabix_file) as fh:
        cols = fh.readline().decode().rstrip().split('\t')
    
    df = pd.DataFrame(rows, columns=cols)  # Add correct column names
    if file_type == 'leafcutter':
        df['gid'] = df.pid.apply(lambda x: x.split(':')[3])
    elif file_type == 'expression':
        df['gid'] = df.pid.apply(lambda x: x.split('.')[0])
        
    return df


In [15]:
# bigwig_list = pd.read_csv('../code/results/ds_sashimi_plots/input/GTEx.bigwig_list.tsv', sep='\t')
# bigwig_list['sample_id'] = bigwig_list['sample'].apply(lambda x: '-'.join(x.split('-')[:2]))
# bigwig_list['sample_name'] = bigwig_list.Group_label + '.' + bigwig_list.sample_id
# bigwig_list = bigwig_list[['sample_name', 'bigwigPath', 'Group_label', 'strand']]
# bigwig_list.columns = ['sample', 'bigwigPath', 'Group_label', 'strand']
# bigwig_list.to_csv('../code/results/ds_sashimi_plots/input/GTEx.bigwig_list.renamed.tsv', sep='\t', header=True, index=False)

In [45]:
ten_tissues = ['Brain-Anteriorcingulatecortex_BA24',
 'Brain-Cortex',
 'Brain-FrontalCortex_BA9',
 'Brain-Putamen_basalganglia',
 'Heart-AtrialAppendage',
 'Liver',
 'Lung',
 'Muscle-Skeletal',
 'Skin-NotSunExposed_Suprapubic',
 'WholeBlood']
run_tabix(f'../code/results/pheno/GTEx/{ten_tissues[9]}/leafcutter.phen_chr1.sorted.for_links.bed.gz', 'chr1', 954081, 955923, file_type = '').reset_index(drop=True)

,#Chr,start,end,pid,gid,strand,GTEX-111YS,GTEX-1122O,GTEX-1128S,GTEX-113IC,...,GTEX-ZV7C,GTEX-ZVE2,GTEX-ZVP2,GTEX-ZVT2,GTEX-ZVT3,GTEX-ZVT4,GTEX-ZVZP,GTEX-ZVZQ,GTEX-ZXES,GTEX-ZXG5
0,chr1,954082,955922,chr1:954082:955922:clu_1921_-:PR,.,-,92.75537309828545,71.71145685997172,69.99699969996999,92.38385376999238,...,95.04950495049505,75.24752475247524,86.97238144867119,100.0,92.07920792079209,94.175888177053,86.79867986798679,96.33296662999633,88.351776354106,100.0
1,chr1,954523,955922,chr1:954523:955922:clu_1921_-:UP,.,-,8.234725911615552,29.278642149929272,30.993099309930994,8.606245239908606,...,5.9405940594059405,25.742574257425744,14.017717561229807,0.9900990099009901,8.91089108910891,6.814210832847991,14.19141914191419,4.657132379904657,12.63832265579499,0.9900990099009901


In [80]:
import re
def CreateBedForSashimiLinks(tissue_list, intron_list):
    chrom = intron_list[0].split(':')[0]
    start_list = [int(x.split(':')[1]) for x in intron_list]
    end_list = [int(x.split(':')[2]) for x in intron_list]

    intron_list = [re.sub(r":clu_[^:]*:", ":", x) for x in intron_list]

    start = np.min(start_list) - 1
    end = np.max(end_list) + 1

    counter = 0

    for tissue in tissue_list:
        tissue_introns = run_tabix(f'../code/results/pheno/GTEx/{tissue}/leafcutter.phen_{chrom}.sorted.for_links.bed.gz', 
                  chrom, start, end, file_type = '')
        bed_cols = list(tissue_introns.columns[:6])
        tissue_introns.index = tissue_introns.pid.apply(lambda x:re.sub(r":clu_[^:]*:", ":", x))
        tissue_introns = tissue_introns.loc[intron_list]
        tissue_introns = tissue_introns.reset_index(drop=True)
        tissue_introns.columns = bed_cols + [f'{tissue}.{x}' for x in tissue_introns.columns[6:]]
        tissue_introns.pid = tissue_introns.pid.apply(lambda x:re.sub(r":clu_[^:]*:", ":", x))
        if counter > 0:
            introns_bed = introns_bed.merge(tissue_introns, left_on=bed_cols, right_on=bed_cols)
        else:
            introns_bed = tissue_introns

        counter += 1

    introns_bed.start = introns_bed.start.astype(int)
    introns_bed.end = introns_bed.end.astype(int) -1
            
    return introns_bed



In [87]:
links_prefix = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_links'
intron_list = ['chr1:954082:955922:clu_2407_-:PR', 'chr1:954523:955922:clu_2407_-:UP']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/NOC2L.bed', sep='\t', header=True, index=False)

intron_list = ['chr17:69250621:69251746:clu_39547_-:PR', 'chr17:69250621:69253572:clu_39547_-:UP', 'chr17:69251866:69253572:clu_39547_-:PR']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/ABCA5.bed', sep='\t', header=True, index=False)

intron_list = ['chr1:19251863:19254781:clu_308_+:PR', 'chr1:19252248:19254781:clu_308_+:UP']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/MRTO4.bed', sep='\t', header=True, index=False)

intron_list = ['chr2:96829134:96831968:clu_5250_+:UP', 'chr2:96829134:96832551:clu_5250_+:PR', 'chr2:96832068:96832551:clu_5250_+:UP']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/CNNM3.bed', sep='\t', header=True, index=False)

intron_list = ['chr6:166330247:166331384:clu_17021_-:UP',
               'chr6:166330247:166342418:clu_17021_-:PR',
               'chr6:166331407:166342418:clu_17021_-:UP']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/SFT2D1.bed', sep='\t', header=True, index=False)

intron_list = ['chr6:29608733:29609228:clu_16194_-:PR',
               'chr6:29608733:29610923:clu_16194_-:UP',
               'chr6:29609379:29610923:clu_16194_-:PR']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/GABBR1.bed', sep='\t', header=True, index=False)


intron_list = ['chr17:7191358:7191892:clu_38502_-:PR',
                'chr17:7191358:7192944:clu_38502_-:UP',
                'chr17:7192002:7192944:clu_38502_-:PR']
introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/DLG4.bed', sep='\t', header=True, index=False)

intron_list = ['chr3:186784696:186784961:clu_9368_+:PR',
                'chr3:186784696:186785882:clu_9368_+:UP',
                'chr3:186785101:186785882:clu_9368_+:PR']

introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/EIF4A2.bed', sep='\t', header=True, index=False)




intron_list = ['chr8:2124217:2129126:clu_19481_+:PR',
                'chr8:2127881:2129126:clu_19481_+:UP']

introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/MYOM2.bed', sep='\t', header=True, index=False)

intron_list = ['chr6:36598983:36599820:clu_15382_+:UP',
                'chr6:36598983:36601151:clu_15382_+:PR',
                'chr6:36600276:36601151:clu_15382_+:UP']

introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/SRSF3.bed', sep='\t', header=True, index=False)

intron_list = ['chr19:15397855:15398629:clu_42835_-:UP',
                'chr19:15397855:15399301:clu_42835_-:PR',
                'chr19:15398703:15399301:clu_42835_-:UP',
                'chr19:15398766:15399301:clu_42835_-:UP']

introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/AKAP8L.bed', sep='\t', header=True, index=False)

In [ ]:
introns_list = ['chr19:15397855:15398629:clu_42835_-:UP',
                'chr19:15397855:15399301:clu_42835_-:PR',
                'chr19:15398703:15399301:clu_42835_-:UP',
                'chr19:15398766:15399301:clu_42835_-:UP']

introns_bed = CreateBedForSashimiLinks(ten_tissues, intron_list)
introns_bed.to_csv(f'{links_prefix}/AKAP8L.bed', sep='\t', header=True, index=False)

In [53]:
intron_list = ['chr17:69250621:69251746:clu_39547_-:PR', 'chr17:69250621:69253572:clu_39547_-:UP', 'chr17:69251866:69253572:clu_39547_-:PR']
CreateBedForSashimiLinks(ten_tissues, intron_list)

,#Chr,start,end,pid,gid,strand,Brain-Anteriorcingulatecortex_BA24.GTEX-11DZ1,Brain-Anteriorcingulatecortex_BA24.GTEX-11GSO,Brain-Anteriorcingulatecortex_BA24.GTEX-11GSP,Brain-Anteriorcingulatecortex_BA24.GTEX-11UD1,...,WholeBlood.GTEX-ZV7C,WholeBlood.GTEX-ZVE2,WholeBlood.GTEX-ZVP2,WholeBlood.GTEX-ZVT2,WholeBlood.GTEX-ZVT3,WholeBlood.GTEX-ZVT4,WholeBlood.GTEX-ZVZP,WholeBlood.GTEX-ZVZQ,WholeBlood.GTEX-ZXES,WholeBlood.GTEX-ZXG5
0,chr17,69250621,69251746,chr17:69250621:69251746:PR,.,-,55.24788476330327,51.43204056853082,59.86085094995987,55.995599559956,...,60.396039603960396,50.495049504950494,24.28654630168899,44.99449944994499,46.686976389946686,36.993699369937,60.396039603960396,36.993699369937,53.10057321521625,28.71287128712871
1,chr17,69250621,69253572,chr17:69250621:69253572:UP,.,-,0.7062443185791204,2.864081137061638,0.9900990099009901,0.9900990099009901,...,20.792079207920793,13.366336633663368,47.58299359347699,22.992299229922992,23.838537699923837,27.992799279927993,10.891089108910892,27.992799279927993,11.412193850964043,36.633663366336634
2,chr17,69251866,69253572,chr17:69251866:69253572:PR,.,-,44.059855954129084,45.71816298835798,41.12924805994113,44.99449944994499,...,20.792079207920793,38.11881188118812,30.110658124635993,33.993399339933994,31.454683929931456,36.993699369937,30.693069306930692,36.993699369937,37.46743095362168,36.633663366336634


In [73]:
def format_coordinates(s):
    return re.sub(r'(:|\-)(\d+)', lambda m: f"{m.group(1)}{int(m.group(2)):,}", s)

# Example usage


def prepareGenomeTracks(intron_list, plot_name, window_ext = 200,
                        bw_list = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/input/GTEx.bigwig_list.renamed.tsv', 
                        out_prefix = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_tracks/',
                        groups_settings = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/input/GTEx.GroupsFile.tsv',
                        genes_track = '/project/yangili1/cfbuenabadn/AggregateGenometracks/PremadeTracks/gencode.v26.FromGTEx.genes.bed12.gz',
                        links_prefix = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_links/',
                        plot_prefix = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_plots/'):

    tissue_list = ['Brain-Anteriorcingulatecortex_BA24',
                     'Brain-Cortex',
                     'Brain-FrontalCortex_BA9',
                     'Brain-Putamen_basalganglia',
                     'Heart-AtrialAppendage',
                     'Liver',
                     'Lung',
                     'Muscle-Skeletal',
                     'Skin-NotSunExposed_Suprapubic',
                     'WholeBlood']

    sashimi_bed = links_prefix + plot_name + '.bed.gz'

    chrom = intron_list[0].split(':')[0]
    start_list = [int(x.split(':')[1]) for x in intron_list]
    end_list = [int(x.split(':')[2]) for x in intron_list]

    start = str(np.min(start_list) - window_ext)
    end = str(np.max(end_list) + window_ext)
    region = format_coordinates(f'{chrom}:{start}-{end}')

    out_prefix = out_prefix + plot_name + '.'

    plot_file = plot_prefix + plot_name + '.png'
    
    pgt_cmd = f'''
    python AggregateBigwigsForPlotting.py --BigwigList {bw_list} --BigwigListType KeyFile --OutputPrefix {out_prefix} --Region {region}  --GroupSettingsFile {groups_settings} --Bed12GenesToIni {genes_track} --BedfileForSashimiLinks {sashimi_bed} --TracksTemplate /project/yangili1/cfbuenabadn/AggregateGenometracks/tracks_templates/GeneralPurposeDS.ini
    
    pyGenomeTracks  --region {region} --tracks {out_prefix}tracks.ini -o {plot_file}
    '''

    print(pgt_cmd)

In [74]:
'ABCA5', 'CNNM3', 'DLG4', 'GABBR1', 'MRTO4', 'NOC2L', 'SFT2D1'

('ABCA5', 'CNNM3', 'DLG4', 'GABBR1', 'MRTO4', 'NOC2L', 'SFT2D1')

In [85]:
links_prefix = '/project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_links'
intron_list = ['chr1:954082:955922:clu_2407_-:PR', 'chr1:954523:955922:clu_2407_-:UP']
gene_name = 'NOC2L'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr17:69250621:69251746:clu_39547_-:PR', 'chr17:69250621:69253572:clu_39547_-:UP', 
               'chr17:69251866:69253572:clu_39547_-:PR']
gene_name = 'ABCA5'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr1:19251863:19254781:clu_308_+:PR', 'chr1:19252248:19254781:clu_308_+:UP']
gene_name = 'MRTO4'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr2:96829134:96831968:clu_5250_+:UP', 'chr2:96829134:96832551:clu_5250_+:PR', 'chr2:96832068:96832551:clu_5250_+:UP']
gene_name = 'CNNM3'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr6:166330247:166331384:clu_17021_-:UP',
               'chr6:166330247:166342418:clu_17021_-:PR',
               'chr6:166331407:166342418:clu_17021_-:UP']
gene_name = 'SFT2D1'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr6:29608733:29609228:clu_16194_-:PR',
               'chr6:29608733:29610923:clu_16194_-:UP',
               'chr6:29609379:29610923:clu_16194_-:PR']
gene_name = 'GABBR1'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr17:7191358:7191892:clu_38502_-:PR',
                'chr17:7191358:7192944:clu_38502_-:UP',
                'chr17:7192002:7192944:clu_38502_-:PR']
gene_name = 'DLG4'
prepareGenomeTracks(intron_list, gene_name)


intron_list = ['chr3:186784696:186784961:clu_9368_+:PR',
                'chr3:186784696:186785882:clu_9368_+:UP',
                'chr3:186785101:186785882:clu_9368_+:PR']
gene_name = 'EIF4A2'
prepareGenomeTracks(intron_list, gene_name)







intron_list = ['chr8:2124217:2129126:clu_19481_+:PR',
                'chr8:2127881:2129126:clu_19481_+:UP']

gene_name = 'MYOM2'
prepareGenomeTracks(intron_list, gene_name)

intron_list = ['chr6:36598983:36599820:clu_15382_+:UP',
                'chr6:36598983:36601151:clu_15382_+:PR',
                'chr6:36600276:36601151:clu_15382_+:UP']

gene_name = 'SRSF3'
prepareGenomeTracks(intron_list, gene_name)


intron_list = ['chr19:15397855:15398629:clu_42835_-:UP',
                'chr19:15397855:15399301:clu_42835_-:PR',
                'chr19:15398703:15399301:clu_42835_-:UP',
                'chr19:15398766:15399301:clu_42835_-:UP']

gene_name = 'AKAP8L'
prepareGenomeTracks(intron_list, gene_name)



    python AggregateBigwigsForPlotting.py --BigwigList /project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/input/GTEx.bigwig_list.renamed.tsv --BigwigListType KeyFile --OutputPrefix /project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_tracks/NOC2L. --Region chr1:953,882-956,122  --GroupSettingsFile /project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/input/GTEx.GroupsFile.tsv --Bed12GenesToIni /project/yangili1/cfbuenabadn/AggregateGenometracks/PremadeTracks/gencode.v26.FromGTEx.genes.bed12.gz --BedfileForSashimiLinks /project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_links/NOC2L.bed.gz --TracksTemplate /project/yangili1/cfbuenabadn/AggregateGenometracks/tracks_templates/GeneralPurposeDS.ini
    
    pyGenomeTracks  --region chr1:953,882-956,122 --tracks /project/yangili1/cfbuenabadn/leafcutter2_paper/code/results/ds_sashimi_plots/ds_tracks/NOC2L.tracks.ini -o /project/yangili1

In [68]:
def format_coordinates(s):
    return re.sub(r'(:|\-)(\d+)', lambda m: f"{m.group(1)}{int(m.group(2)):,}", s)

# Example usage
s = 'chr6:29608533-29611123'
format_coordinates(s)

'chr6:29,608,533-29,611,123'